In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers
!pip install -q gradio

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import gradio as gr

In [ ]:
# Small knowledge base / sample query
documents = [
    "Python is a versatile high-level programming language created by Guido van Rossum in 1991. It emphasizes readability and simplicity, with code that reads almost like natural language. Born in 1991, Python has grown to serve developers across the land, helping engineers build systems for various applications. From data science to web apps, Python works quietly until the job is done, known for its simple syntax yet powerful capabilities.",
    "RAG stands for Retrieval-Augmented Generation.",
    "Hugging Face provides NLP tools and transformers.",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is a rapidly expanding field. Machine learning is a subset of AI, focusing on systems that learn from data. Deep learning is a further subset of machine learning that uses neural networks with multiple layers, enabling it to learn complex patterns and often powers generative AI, which can create new text, images, and other media. Natural Language Processing deals with the interaction between computers and human language, contributing significantly to AI by enabling machines to understand, interpret, and generate human language.",
    "The capital of France is Paris.",
    "The Eiffel Tower is located in Paris, France."
]

query = "What is RAG?"

In [ ]:

# Make all documents lowercase for better matching
documents = [doc.lower() for doc in documents]

# Make the query lowercase too
query = query.lower()

In [ ]:

# 1️ Create embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents)

# 2️ Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# Load generator model
model_name = "google/flan-t5-large" # Upgrading from flan-t5-small
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:

def ask(question, k=1, max_length=200):
    question_embedding = embedder.encode([question])
    _, indices = index.search(np.array(question_embedding), k=k)

    # Retrieve multiple documents and join them
    retrieved_docs = [documents[idx] for idx in indices[0]]
    retrieved_context = " ".join(retrieved_docs)

    # Final modified prompt to encourage narrative description
    prompt = f"Context: {retrieved_context}\nQuestion: {question}\nTask: From the provided context, generate a detailed and comprehensive description answering the question, including any relationships between concepts. Ensure the answer is entirely based on the context."
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(**inputs, max_length=max_length)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer


In [ ]:

# Test 1
question = "What does Hugging Face provide in AI?"
answer = ask(question)
print(f"Question: {question}")
print(f"Answer: {answer}")

# Test 2 (longer)
question = "Can you describe Python, including its origins, key characteristics, and common applications, especially concerning its role in AI and data science?"
answer = ask(question, k=1, max_length=500)
print(f"Question: {question}")
print(f"Answer: {answer}")

In [ ]:
def rag_interface_ask(question, k_value=1, max_length_value=500):
    # Lowercase the question if your preprocessing expects it
    question = question.lower()
    return ask(question, k=int(k_value), max_length=int(max_length_value))


In [ ]:

iface = gr.Interface(
    fn=rag_interface_ask,
    inputs=[
        gr.Textbox(lines=2, placeholder="Enter your question here...", label="Question"),
        gr.Slider(minimum=1, maximum=5, step=1, value=1, label="Number of documents to retrieve (k)"),
        gr.Slider(minimum=100, maximum=1000, step=50, value=500, label="Maximum answer length")
    ],
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="RAG Question Answering System",
    description="Ask questions to the RAG system based on its knowledge base. Adjust retrieval (k) and answer length (max_length) for more detailed responses."
)

iface.launch(share=True)